# U08_C1 — ORFs, GC Content, and Codon Usage
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 8:** Predictive Methods using Nucleotide Sequence
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L21–L22, Practical S9 (ORFfinder & EMBOSS cusp)
**Before this:** U07_C1

## 1. What you will learn
- **Explain** the six reading frames of double-stranded DNA [LO6].
- **Identify** Open Reading Frames (ORFs) using start (ATG) and stop (TAA, TAG, TGA) codons with a length threshold [LO6].
- **Calculate** sliding-window GC content to identify CpG islands and compositional shifts [LO6].
- **Quantify** codon usage bias in protein-coding genes [LO6].


## 2. Biological question
When a new genome or cDNA sequence is determined, how do computational pipelines locate the protein-coding genes without prior laboratory evidence? Because stop codons occur by chance once every 21 codons in random DNA ($3 / 64 \approx 1/21$), an unusually long stretch without a stop codon starting with ATG strongly suggests a functional protein-coding gene. How do we scan all six reading frames of human *HBB* mRNA to pinpoint its true coding sequence?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U08_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The Six Reading Frames
### Theory
Because the genetic code is read in non-overlapping triplets of nucleotides (codons), any DNA sequence has three reading frames on the forward strand depending on whether translation begins at index 0 (+1), index 1 (+2), or index 2 (+3).

Because DNA is double-stranded and transcription can occur on the opposite complementary strand, there are also three reading frames on the reverse complement strand (-1, -2, -3). Thus, any genomic DNA locus has **six possible reading frames**.


In [ ]:
# Worked Example: Translating in all six reading frames
from Bio import SeqIO

hbb_path = DATA_DIR / "sequences" / "hbb_human_mrna.fasta"
record = SeqIO.read(hbb_path, "fasta")
dna = record.seq

frames = {}
for i in range(3):
    frames[f"+{i+1}"] = dna[i:].translate()

rc = dna.reverse_complement()
for i in range(3):
    frames[f"-{i+1}"] = rc[i:].translate()

print("=== 6-Frame Translation of HBB mRNA ===")
for frame_name, peptide in frames.items():
    stop_count = peptide.count("*")
    print(f"Frame {frame_name}: Length = {len(peptide)} aa | Stop Codons (*) = {stop_count:2} | Start: {peptide[:20]}...")


## 4. Concept 2: Identifying Open Reading Frames (ORFs)
### Theory
An **Open Reading Frame (ORF)** is a continuous stretch of codons beginning with a start codon (canonical `ATG`), containing sense codons, and terminating at an in-frame stop codon (`TAA`, `TAG`, `TGA`).
- Random non-coding DNA has an expected stop codon every $64 / 3 \approx 21.3$ codons.
- True eukaryotic and bacterial proteins typically contain $>100$ amino acids (300 bp).
- Applying a minimum length threshold (e.g. $\ge 100$ aa) filters out false-positive random stop-free intervals.


In [ ]:
# Worked Example: Scanning for genuine ORFs in HBB mRNA
def find_forward_orfs(seq_str, min_aa=50):
    seq_str = seq_str.upper()
    orfs = []
    for frame in range(3):
        for i in range(frame, len(seq_str) - 2, 3):
            if seq_str[i:i+3] == "ATG":
                for j in range(i + 3, len(seq_str) - 2, 3):
                    if seq_str[j:j+3] in ["TAA", "TAG", "TGA"]:
                        aa_len = (j + 3 - i - 3) // 3
                        if aa_len >= min_aa:
                            orfs.append({
                                "frame": frame + 1,
                                "start_1based": i + 1,
                                "end_1based": j + 3,
                                "length_nt": j + 3 - i,
                                "length_aa": aa_len,
                                "subseq": seq_str[i:j+3]
                            })
                        break
    return orfs

found_orfs = find_forward_orfs(str(dna), min_aa=50)
print(f"Found {len(found_orfs)} candidate ORFs >= 50 aa:")
for idx, orf in enumerate(found_orfs, 1):
    print(f"  ORF {idx}: Frame +{orf['frame']} | 1-based Coordinates: {orf['start_1based']}..{orf['end_1based']} "
          f"| Length: {orf['length_nt']} nt ({orf['length_aa']} aa)")


## 5. Concept 3: Sliding-Window GC Content
### Theory
GC content varies locally across chromosomes. Promoters often contain unmethylated **CpG islands** ($>50\%$ GC and observed-to-expected CpG ratio $>0.6$). Exons frequently exhibit elevated GC content compared to surrounding introns. A **sliding window** moves across the sequence with step size $S$ and window size $W$, calculating local GC percentage.


In [ ]:
# Worked Example: Sliding window GC profile
import matplotlib.pyplot as plt

window_size = 50
step = 10
dna_str = str(dna)
gc_profile = []
positions = []

for i in range(0, len(dna_str) - window_size + 1, step):
    window = dna_str[i:i + window_size]
    gc_val = ((window.count("G") + window.count("C")) / window_size) * 100
    gc_profile.append(gc_val)
    positions.append(i + window_size // 2)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(positions, gc_profile, color="teal", lw=1.5)
ax.axhline(51.8, color="red", linestyle="--", alpha=0.7, label="Mean GC% (51.8%)")
ax.axvspan(51, 494, color="yellow", alpha=0.2, label="Coding CDS (51-494)")
ax.set_xlabel("Nucleotide Position in HBB mRNA")
ax.set_ylabel("GC Percentage")
ax.set_title("Sliding Window GC Profile (Window = 50 bp, Step = 10 bp)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT_DIR / "hbb_gc_profile.png", dpi=100)
plt.show()


## 6. Concept 4: Codon Usage Bias
### Theory
The genetic code has 64 codons encoding 20 amino acids and 3 stops. Most amino acids are specified by 2 to 6 synonymous codons. Organisms do *not* use synonymous codons equally:
- Highly expressed genes preferentially use codons recognized by the most abundant cellular tRNAs (**translational efficiency**).
- Heterologous expression (e.g. human insulin in *E. coli*) often requires codon optimization to match host codon preferences.


In [ ]:
# Worked Example: Computing codon counts for HBB coding sequence
cds_str = str(found_orfs[0]["subseq"])
codon_counts = {}

for i in range(0, len(cds_str) - 2, 3):
    codon = cds_str[i:i+3]
    codon_counts[codon] = codon_counts.get(codon, 0) + 1

# Inspect Leucine codons (encoded by 6 codons: CTT, CTC, CTA, CTG, TTA, TTG)
leu_codons = ["CTT", "CTC", "CTA", "CTG", "TTA", "TTG"]
print("Leucine Synonymous Codon Usage in Human HBB CDS:")
for lc in leu_codons:
    cnt = codon_counts.get(lc, 0)
    print(f"  {lc} : {cnt:2} occurrences")


## 7. Compare with NCBI ORFfinder
In Practical S9, we pasted the HBB mRNA sequence into NCBI ORFfinder. The primary returned ORF was:
- Frame: `+3` (or 1-based coordinates 51..494)
- Length: `444 nt` / `147 aa`
Our Python code produces the exact same start position (51), end position (494), and translated protein length.


## 8. Summary
- Double-stranded DNA has 6 reading frames (3 forward, 3 reverse complement).
- An ORF starts with ATG and ends at an in-frame stop codon (TAA, TAG, TGA).
- Length thresholds (e.g. $>300$ nt) eliminate the high background of random stop-free segments.
- Codon bias reflects evolutionary optimization for translational accuracy and efficiency.


## 9. Next Steps
- Complete practice exercises in `U08_P_Nucleotide_Analysis.ipynb`.
- Self-test ORF finding and codon bias concepts in `U08_Q_Predictive_Nucleotide.ipynb`.
